# Module 02: Basic Orbit Simulation

In this module we build a complete **two-body orbital simulation** from scratch — a spacecraft orbiting Earth under point-mass gravity. This is the canonical "Hello World" of Basilisk.

### Learning Objectives
- Define an orbit using classical orbital elements (COEs)
- Configure `SpacecraftContainer` and `gravityEffector`
- Record and plot position/velocity data
- Verify orbital period using Kepler's third law

---

## Background: Classical Orbital Elements (COEs)

| Element | Symbol | Description |
|---|---|---|
| Semi-major axis | `a` | Average orbital radius |
| Eccentricity | `e` | Shape (0 = circle, <1 = ellipse) |
| Inclination | `i` | Tilt relative to reference plane |
| RAAN | `Ω` | Right Ascension of Ascending Node |
| Arg. of Perigee | `ω` | Rotation of ellipse in orbital plane |
| True Anomaly | `ν` | Spacecraft position along orbit |

Basilisk's `orbitalMotion.ClassicElements` struct holds these, and `orbitalMotion.elem2rv()` converts them to position/velocity vectors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from Basilisk.utilities import SimulationBaseClass
from Basilisk.utilities import macros
from Basilisk.utilities import orbitalMotion
from Basilisk.utilities import unitTestSupport
from Basilisk.simulation import spacecraft
from Basilisk.simulation import gravityEffector
from Basilisk.architecture import bskLogging

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)  # suppress info output
print("Imports OK.")

---

## Step 1 — Define the Orbit

In [ ]:
# ── Physical constants ────────────────────────────────────────────────────────
mu_earth = 3.986004418e14   # Earth gravitational parameter [m^3/s^2]
R_earth  = 6371e3           # Earth mean radius [m]

# ── Define classical orbital elements ─────────────────────────────────────────
oe = orbitalMotion.ClassicElements()
oe.a     = R_earth + 500e3   # semi-major axis: 500 km altitude circular orbit
oe.e     = 0.001             # nearly circular
oe.i     = 45.0 * macros.D2R # 45 degree inclination
oe.Omega = 48.2 * macros.D2R # RAAN
oe.omega = 347.8 * macros.D2R# argument of perigee
oe.f     = 85.3 * macros.D2R # true anomaly (starting position along orbit)

# ── Convert to inertial position & velocity ───────────────────────────────────
r0, v0 = orbitalMotion.elem2rv(mu_earth, oe)

# ── Compute expected orbital period (Kepler's 3rd law) ────────────────────────
T_orbit = 2 * np.pi * np.sqrt(oe.a**3 / mu_earth)

print(f"Orbital altitude:     {(oe.a - R_earth)/1e3:.1f} km")
print(f"Semi-major axis:      {oe.a/1e3:.2f} km")
print(f"Orbital period:       {T_orbit/60:.2f} min  ({T_orbit:.1f} s)")
print(f"Initial position (km): [{r0[0]/1e3:.2f}, {r0[1]/1e3:.2f}, {r0[2]/1e3:.2f}]")
print(f"Initial velocity (m/s): [{v0[0]:.2f}, {v0[1]:.2f}, {v0[2]:.2f}]")

---

## Step 2 — Build the Simulation

In [ ]:
# ── Create simulation skeleton ────────────────────────────────────────────────
scSim = SimulationBaseClass.SimBaseClass()
dynProcess = scSim.CreateNewProcess("DynamicsProcess")
simTimeStep = macros.sec2nano(10.0)   # 10-second dynamics time step
dynTask = scSim.CreateNewTask("DynamicsTask", simTimeStep)
dynProcess.addTask(dynTask)

# ── Create the Spacecraft object ──────────────────────────────────────────────
scObject = spacecraft.Spacecraft()
scObject.ModelTag = "MySpacecraft"    # human-readable identifier

# Set inertia matrix (kg*m^2) — doesn't matter for point-mass orbit
I = [900., 0., 0.,
     0., 800., 0.,
     0., 0., 600.]
scObject.hub.mHub       = 750.0   # total mass [kg]
scObject.hub.r_BcB_B    = [[0.0], [0.0], [0.0]]  # CoM offset from hub frame
scObject.hub.IHubPntBc_B = unitTestSupport.np2EigenMatrix3d(np.array(I).reshape(3,3))

# ── Set initial conditions (state vector) ─────────────────────────────────────
scObject.hub.r_CN_NInit = r0.tolist()    # initial position  [m]
scObject.hub.v_CN_NInit = v0.tolist()    # initial velocity  [m/s]

print("Spacecraft created:", scObject.ModelTag)

In [ ]:
# ── Create the gravity effector ───────────────────────────────────────────────
gravFactory = gravityEffector.GravBodyDataVector()
earth = gravFactory.createEarth()
earth.isCentralBody = True

# Attach gravity to the spacecraft
scObject.gravField.gravBodies = gravFactory

# ── Add spacecraft to the dynamics task ───────────────────────────────────────
scSim.AddModelToTask("DynamicsTask", scObject)

print("Gravity effector attached.")

---

## Step 3 — Set Up Data Logging

In [ ]:
# ── Attach recorders to the spacecraft state output messages ──────────────────
# scObject.scStateOutMsg publishes position, velocity, attitude each time step
dataRec = scObject.scStateOutMsg.recorder()
scSim.AddModelToTask("DynamicsTask", dataRec)

print("Data recorder attached to scStateOutMsg.")

---

## Step 4 — Initialize & Run

In [ ]:
# ── Initialize and run for 1 full orbital period ──────────────────────────────
scSim.InitializeSimulation()

simStopTime = macros.sec2nano(T_orbit)   # exactly one orbit
scSim.ConfigureStopTime(simStopTime)
scSim.ExecuteSimulation()

print(f"Simulation complete. Final time: {scSim.TotalSim.CurrentNanos * macros.NANO2SEC:.1f} s")

---

## Step 5 — Extract & Analyze Data

In [ ]:
# ── Read logged data ──────────────────────────────────────────────────────────
t   = dataRec.times() * macros.NANO2SEC   # time in seconds
pos = dataRec.r_BN_N                       # position vectors  [N x 3]  (m)
vel = dataRec.v_BN_N                       # velocity vectors  [N x 3]  (m/s)

# Orbital radius over time
r_norm = np.linalg.norm(pos, axis=1) / 1e3   # [km]
v_norm = np.linalg.norm(vel, axis=1)          # [m/s]

print(f"Recorded {len(t)} data points.")
print(f"Position range: {r_norm.min():.3f} — {r_norm.max():.3f} km")
print(f"Velocity range: {v_norm.min():.3f} — {v_norm.max():.3f} m/s")

# Verify return to start
pos_error = np.linalg.norm(pos[-1] - pos[0]) / 1e3
print(f"\nPosition error after 1 orbit: {pos_error:.4f} km  (should be ~0)")

In [ ]:
# ── Plot 1: Orbital radius and speed vs time ──────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(t / 60, r_norm, color='steelblue')
axes[0].axhline(oe.a / 1e3, color='red', linestyle='--', label=f'Semi-major axis = {oe.a/1e3:.1f} km')
axes[0].set_ylabel('Orbital Radius (km)')
axes[0].set_title('Spacecraft Orbit — 500 km Circular Earth Orbit')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t / 60, v_norm, color='darkorange')
axes[1].set_xlabel('Time (min)')
axes[1].set_ylabel('Speed (m/s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/02_orbit_radius_speed.png', dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved.")

In [ ]:
import os
os.makedirs('plots', exist_ok=True)

# ── Plot 2: 3-D orbital track ─────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot(pos[:, 0]/1e6, pos[:, 1]/1e6, pos[:, 2]/1e6,
        color='steelblue', linewidth=1.5, label='Orbit track')
ax.scatter(*pos[0]/1e6, color='green', s=80, zorder=5, label='Start')
ax.scatter(*pos[-1]/1e6, color='red', s=80, zorder=5, label='End')

# Draw Earth sphere
u, v = np.mgrid[0:2*np.pi:60j, 0:np.pi:30j]
xs = (R_earth/1e6) * np.cos(u) * np.sin(v)
ys = (R_earth/1e6) * np.sin(u) * np.sin(v)
zs = (R_earth/1e6) * np.cos(v)
ax.plot_surface(xs, ys, zs, color='royalblue', alpha=0.25)

ax.set_xlabel('X (Mm)')
ax.set_ylabel('Y (Mm)')
ax.set_zlabel('Z (Mm)')
ax.set_title('3D Orbit Track (1 full orbit)')
ax.legend()
plt.tight_layout()
plt.savefig('plots/02_orbit_3d.png', dpi=100, bbox_inches='tight')
plt.show()

---

## Step 6 — Verify Energy Conservation

In [ ]:
# Specific orbital energy should be constant: E = v^2/2 - mu/r
E = 0.5 * v_norm**2 - mu_earth / (r_norm * 1e3)

E_theory = -mu_earth / (2 * oe.a)   # theoretical value
E_error  = (E - E_theory) / abs(E_theory) * 100

print(f"Theoretical specific energy: {E_theory:.4f} J/kg")
print(f"Simulated  specific energy:  {E.mean():.4f} ± {E.std():.6f} J/kg")
print(f"Max relative error:          {abs(E_error).max():.2e} %")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t / 60, E_error, color='purple')
ax.axhline(0, color='gray', linestyle='--')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Energy Error (%)')
ax.set_title('Specific Orbital Energy Conservation')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/02_energy_conservation.png', dpi=100, bbox_inches='tight')
plt.show()

---

## Exercise: Try These Variations

1. **Change the orbit**: Set `oe.e = 0.5` for a highly elliptical orbit. Observe how speed varies.
2. **Change the altitude**: Try 200 km (LEO) and 35786 km (GEO). Compare periods.
3. **Vary the time step**: Use 1 s vs 60 s. How does energy conservation change?
4. **Add J2 perturbation**: Use `gravityEffector.loadGravFromFile()` to add oblateness effects.

---

## Key Takeaways

- Use `orbitalMotion.ClassicElements` + `elem2rv()` to set initial conditions
- `spacecraft.Spacecraft` is the central dynamics object
- `gravityEffector.GravBodyDataVector` attaches gravity — call `.createEarth()` (or Moon, Mars, …)
- Attach recorders with `.recorder()` and add them to a task
- After simulation, access data as NumPy arrays: `dataRec.r_BN_N`, `dataRec.v_BN_N`

**Next: [03 - Spacecraft Dynamics](03_spacecraft_dynamics.ipynb)**